In [ ]:
# Residential proximity to oil and gas development and socioeconomic conditions: A comparative analysis of 33 countries
# 02: PopExposure Results Plotting

# This file plots results from Popexposure 
# Author: Lara Schwarz
# Last updated: November 2025
# Reviewed by Nina Flores: September 2025

## Step 1: Load data used for plotting 
## Step 2: Combining and matching datasets
## Figure 1 
## Figure 2 
## Figure 3 
## Figure 4 
## Figure S1 
## Figure S2
## Figure S3
## Figure S4
## Figure S5
## Figure S6   
## Figure S7

In [ ]:
## Importing libraries

import geopandas as gpd
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from thefuzz import process
import pandas as pd
import pathlib
import pandas as pd
import sys
import os
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from rasterio.plot import show
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import fiona
import geopandas as gpd
from shapely.geometry import Polygon
import ee
import geemap
import matplotlib.cm as cm
import matplotlib.colors as colors
from shapely.geometry import box
import math
from shapely.geometry import Point
import glob
import seaborn as sns
import openpyxl
import subprocess
from pathlib import Path
from popexposure import PopEstimator
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import contextily as ctx 
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import csv
import re


In [ ]:
## Defining paths and lists 

repo = pathlib.Path.cwd().parent.parent
share_path = pathlib.Path("/Users/larasch/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/data")

# Define code path and add to sys.path
code_path = repo / "code"

# Define data paths
# location of the oil and gas data
oil_path = share_path / "environmental/oil_and_gas/raw_data/ogim.parquet"

# Define the paths
social_path = repo / "data/IHME_LSAE_admin2_LDIpc_2019_with_loc_id(in)_v2.csv"  

# location of global shapefile
shapefile_path = repo / "data/OneDrive_1_12-11-2024/lbd_standard_admin_2.shp"

# location of the population data
pop_path = share_path / "social_including_census/raw_data/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0.tif"

# Define the directory for the country-specific parquet files
country_dir = share_path / "geo_boundaries/processed_data/countries_ogd"

# Define the directory for the OGIM country-specific parquet files
ogim_dir = share_path / "environmental/oil_and_gas/raw_data/ogim_by_country"

# Define the directory for results
results_dir = repo / "results"

# Folder to export figures
fig_dir = repo / "figures"


In [ ]:
## Step 1: Load data used for plotting 

# Load EIA Excel data
excel_path =  repo / "data/EIA_global_ogd_production.xlsx"
eia_data = pd.read_excel(excel_path)
eia_data.rename(columns=lambda x: x.strip(), inplace=True)  # remove extra spaces in column names

# Load global oil data
oil = gpd.read_parquet(oil_path)

#  Load shapefile
global_shapefile = gpd.read_file(shapefile_path)
countries = global_shapefile.dissolve(by="ADM0_NAME", as_index=False)

## Load global country social data 
try:
    social_data = pd.read_csv(social_path, encoding='ISO-8859-1')  # or try encoding='utf-16'
    print("CSV loaded successfully with encoding ISO-8859-1.")
except UnicodeDecodeError:
    print("Error: Unable to decode CSV with the provided encoding.")

# merge the social and shapefile data
merged_shapefile_social = global_shapefile.merge(social_data, how='left', left_on='loc_id', right_on='loc_id')

# WBG classifications
wbg_path = repo / "data/WBG_classification.xlsx"
wbg_df = pd.read_excel(wbg_path)
wbg_df['Economy'] = wbg_df['Economy'].str.lower().str.replace(" ", "_")

# GRDI Data
GRDI_path = results_dir / "ogd_adm2_avg_GRDI.csv"
GRDI_data = pd.read_csv(GRDI_path)
GRDI_data = GRDI_data.rename(columns={
    'exposed_1km': 'GRDI_1km',
    'exposed_3km': 'GRDI_3km',
    'exposed_5km': 'GRDI_5km'
})
GRDI_data['country'] = GRDI_data['country'].str.lower().str.replace(" ", "_")

# Load country level exposure data
country_exp_path = results_dir / "ogd_country_pop_exposed.csv"
country_exposure = pd.read_csv(country_exp_path)

# Load ADM2 exposure data
exposed_adm2_path = results_dir / "ogd_adm2_pop_exposed.csv"
adm2_exposed = pd.read_csv(exposed_adm2_path)

# Load production data
prod_path = "/Users/larasch/Documents/UCB_postdoc/Research/global_oil_gas/ogd/data/INT-Export-11-12-2025_11-17-42.csv"
#--- Read CSV manually (handles odd formatting) ---
rows = []
with open(prod_path, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    for row in reader:
        rows.append(row)

# --- Extract years from header row ---
years = []
for line in rows:
    if any(re.match(r"^\d{4}$", x.strip()) for x in line):
        years = [x.strip() for x in line if re.match(r"^\d{4}$", x.strip())]
        break

# --- Build DataFrame ---
data = []
for r in rows:
    if not r:  # skip empty rows
        continue
    country = r[1].strip() if len(r) > 1 else None
    if not country:  # skip rows with no country name
        continue
    values = r[2:] if len(r) > 2 else []
    values = (values + [None] * len(years))[:len(years)]  # pad/truncate
    data.append([country] + values)

df_prod = pd.DataFrame(data, columns=["country"] + years)
df_prod[years] = df_prod[years].apply(pd.to_numeric, errors="coerce")


In [ ]:
# Step 2: Combining and matching datasets

# Fuzzy match countries
# Define columns for matching
shapefile_country_col = 'ADM0_NAME'  # adjust to your shapefile column
eia_country_col = 'Country'             # adjust to your Excel column

# Make sure both columns are string type
countries[shapefile_country_col] = countries[shapefile_country_col].astype(str)
eia_data[eia_country_col] = eia_data[eia_country_col].astype(str)

eia_country_list = eia_data[eia_country_col].tolist()

# --- Countries to exclude from matching ---
exclude_countries = {
    "British Indian Ocean Territory",
    "Guinea",
    "South Georgia and the South Sandwich Islands",
    "√É¬Öland"
}

# Create a dictionary mapping shapefile countries -> best match in EIA
mapping = {}
threshold = 90  # adjust as needed

for country in countries[shapefile_country_col].unique():
    if country == "Taiwan (Province of China)":
        mapping[country] = "Taiwan"  # force exact match
    # Skip excluded countries
    elif country in exclude_countries:
        mapping[country] = None
    else:
        match, score = process.extractOne(country, eia_country_list)
        if score >= threshold:
            mapping[country] = match
        else:
            mapping[country] = None  # no confident match
        
# Add a column to the shapefile with matched country names
countries['matched_country'] = countries[shapefile_country_col].map(mapping)

#  Merge using the matched names
# =========================
merged_gdf = countries.merge(
    eia_data,
    how='left',
    left_on='matched_country',
    right_on=eia_country_col
)


# Inspect merged data
print(merged_gdf.head())

In [ ]:
##  Figure 1 

# --- Define high-income countries ---
high_income = [
    "Australia", "Austria", "Belgium", "Canada", "Switzerland", "Germany", "Denmark", "Spain",
    "Finland", "France", "United Kingdom", "Iceland", "Ireland", "Italy", "Japan", "Korea, South",
    "Netherlands", "Norway", "New Zealand", "Portugal", "Singapore", "Sweden", "United States",
    "United Arab Emirates", "Saudi Arabia", "Czechia", "Slovakia", "Slovenia", "Israel",
    "Malta", "Cyprus", "Hong Kong", "Macau", "Brunei", "Bahrain", "Kuwait", "Qatar", "Oman",
    "Luxembourg", "France, Metropolitan", "Greece", "Italy, mainland", "East Germany", "West Germany", "Hawaiian Trade Zone", "Germany, Fed. R. (former)", "Netherlands (Kingd. of the)",
    "USSR (former)", "Czechoslovakia (former)", "Russian Federation"
]

## # --- Ensure both GeoDataFrames use the same CRS ---
merged_gdf = merged_gdf.to_crs(epsg=4326)
oil = oil.to_crs(merged_gdf.crs)

merged_gdf = merged_gdf[
    ~merged_gdf["ADM0_NAME"].str.lower().isin(["antarctica", "greenland"])
]
# --- Separate countries with numeric Percent_global and those with NaN ---
gdf_numeric = merged_gdf[merged_gdf['Percent_global'].notna()]
gdf_na = merged_gdf[merged_gdf['Percent_global'].isna()]

# --- Define color categories ---
def categorize_color(percent):
    if percent < 1:
        return "pink"
    elif 1 <= percent <= 10:
        return "orange"
    else:
        return "red"

# Assign fill color based on Percent_global
gdf_numeric = gdf_numeric.copy()
gdf_numeric["fill_color"] = gdf_numeric["Percent_global"].apply(categorize_color)

# --- Convert to Equal Earth projection (ESRI:54052) ---
equal_earth_crs = "ESRI:54052"
gdf_numeric_ee = gdf_numeric.to_crs(equal_earth_crs)
gdf_na_ee = gdf_na.to_crs(equal_earth_crs)
oil_ee = oil.to_crs(equal_earth_crs)

# --- Optional: Export CSV of countries included on the map ---
# output_csv = results_dir / "countries_on_map.csv"
# country_csv = gdf_numeric[["ADM0_NAME", "matched_country", "Percent_global", "Country", "production"]]  
# country_csv.to_csv(output_csv, index=False)
# print(" Exported CSV: countries_on_map.csv")

# =========================================================
# --- Load UN crude petroleum reserves dataset
# =========================================================
un_path = "/Users/larasch/Documents/UCB_postdoc/Research/global_oil_gas/ogd/data/UNdata_Export_20251117_055406207.csv"
df_un = pd.read_csv(un_path)

# Keep only crude petroleum reserves
df_un = df_un[df_un["Commodity - Transaction"] == "Crude petroleum - reserves"].copy()

# Convert to consistent column names
df_un = df_un.rename(columns={
    "Country or Area": "country",
    "Quantity": "Oil_reserves",
    "Year": "Year"
})

# Convert metric tons → barrels (approx.)
#df_un["Oil_reserves"] = df_un["Oil_reserves"] * 7.33

# Remove rows with missing or flagged quantity
df_un = df_un[df_un["Oil_reserves"].notna()]

# Keep only needed columns
df_un = df_un[["country", "Year", "Oil_reserves"]]

# Convert Year to int
df_un["Year"] = df_un["Year"].astype(int)


# =========================================================
# --- Assign income groups
# =========================================================
df_un["income_group"] = df_un["country"].apply(
    lambda x: "High income" if x in high_income else "Low and middle income"
)

# Keep 1985 onward
df = df_un[df_un["Year"] >= 1990 & (df_un["Year"] <= 2023)]

# Extract unique country values
df_un2 = (
    df_un[['country', 'income_group']]
    .dropna(subset=['country'])
    .drop_duplicates()
    .sort_values('country')
)

# Define output path
output_csv = results_dir / "countries_oil_reserves.csv"

# Export
df_un2.to_csv(output_csv, index=False)

print("Saved:", output_csv)

# =========================================================
# --- Aggregate for panel (b) stacked reserves chart
# =========================================================
df_grouped = df.groupby(["Year", "income_group"], as_index=False)["Oil_reserves"].sum()
df_pivot = df_grouped.pivot(index="Year", columns="income_group", values="Oil_reserves")

# -------------------------------
# --- Create Figure Layout
# -------------------------------
fig = plt.figure(figsize=(16,24))
plt.tight_layout()
spec = gridspec.GridSpec(
    ncols=2, nrows=2,
    height_ratios=[3.2, 1.4],
    wspace=0.25, hspace=0.02,
    figure=fig
)
# --- Subplot a: Map
ax0 = fig.add_subplot(spec[0, :])
gdf_na_ee.plot(ax=ax0, color="lightgray", edgecolor="white", linewidth=0.5, alpha=0.9)
gdf_numeric_ee.plot(ax=ax0, color=gdf_numeric_ee["fill_color"], edgecolor="white", linewidth=0.5, alpha=0.9)
oil_ee.plot(ax=ax0, color="black", markersize=1.5, alpha=0.1)

legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='pink', markersize=10, label='<1%'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='orange', markersize=10, label='1–10%'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='red', markersize=10, label='>10%'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='lightgray', markersize=10, label='No data'),
    Line2D([0], [0], marker='o', color='black', markersize=1.5, linestyle='None', label='Oil and gas wells')
]

ax0.legend(
    handles=legend_elements,
    title="Global production (% global barrels)",
    loc='upper left', bbox_to_anchor=(1.02,1),
    frameon=True, facecolor="white", edgecolor="black", framealpha=1
)
ax0.set_axis_off()
ax0.text(-0.02, 1.02, "a", transform=ax0.transAxes, fontsize=18, fontweight='bold')

# Import data on production

# --- Keep only "World" ---
df_world = df_prod[df_prod["country"].str.strip().str.lower() == "world"]

if df_world.empty:
    raise ValueError("No 'World' entry found in the dataset.")

# --- Reshape to long format for plotting ---
df_world_long = df_world.melt(id_vars=["country"], var_name="year", value_name="production")
df_world_long["year"] = pd.to_numeric(df_world_long["year"], errors="coerce")
df_world_long = df_world_long[(df_world_long["year"] >= 1990) & (df_world_long["year"] <= 2023)]

# ============================================================
# Combined Subplot (b): Production + Reserves
# ============================================================
ax_combined = fig.add_subplot(spec[1, :])

# ---- Left axis: Global production (line)
ax_combined.plot(
    df_world_long["year"],
    df_world_long["production"],
    color="black",
    linewidth=2.5,
    label="Global production"
)
ax_combined.set_xlabel("Year", fontsize=14)
ax_combined.set_ylabel("Global Petroleum Production (Mb/d)", fontsize=12)
ax_combined.set_ylim(30000, 110000)  # <-- change this number as needed

# ---- Right axis: Oil reserves by income group (stacked bars)
ax_right = ax_combined.twinx()
df_pivot = df_pivot[(df_pivot.index >= 1990) & (df_pivot.index <= 2023)]

years = df_pivot.index
hi = df_pivot["High income"].fillna(0)
lmi = df_pivot["Low and middle income"].fillna(0)

bar_width = 0.8

ax_right.bar(
    years,
    hi,
    width=bar_width,
    label="High income reserves",
    alpha=0.6,
    color="#1f77b4"
)

ax_right.bar(
    years,
    lmi,
    width=bar_width,
    bottom=hi,
    label="Low & middle income reserves",
    alpha=0.6,
    color="#ff7f0e"
)
ax_right.set_ylabel("Total Oil Reserves (metric tons)", fontsize=12)
ax_right.set_ylim(0, 350000000)  # <-- change this number as needed

# ---- Combined legend (placed under plot)
lines_left = [Line2D([0], [0], color="black", linewidth=2.5, label="Global production")]
patch_hi = Line2D([0], [0], marker='s', color='w', markerfacecolor='#1f77b4', markersize=10,
                  label='High income reserves')
patch_lmi = Line2D([0], [0], marker='s', color='w', markerfacecolor='#ff7f0e', markersize=10,
                   label='Low & middle income reserves')

ax_combined.legend(
    handles=[lines_left[0], patch_hi, patch_lmi],
    loc="upper left",
    bbox_to_anchor=(0, -0.22),
    ncol=3,
    frameon=False,
    fontsize=12
)

# ---- Panel label
ax_combined.text(-0.04, 1.02, "b", transform=ax_combined.transAxes,
                 fontsize=16, fontweight='bold')

plt.show()

# --- Save figure ---
# Filename
fig_path = os.path.join(fig_dir, "Figure1_global_map_production_reserves.png")

# Save (high resolution, tight layout)
fig.savefig(
    fig_path,
    dpi=600,            # high quality
    bbox_inches="tight",
    pad_inches=0.2
)

print(f"Figure saved to: {fig_path}")

In [ ]:
## Figure 2 (revised, larger text)

# --- Standardize country names for merging ---
social_data['country_location_name'] = social_data['country_location_name'].replace(
    {'Venezuela (Bolivarian Republic of)': 'Venezuela'}
)
social_data['country'] = social_data['country_location_name'].str.lower().str.replace(" ", "_")
country_exposure['country'] = country_exposure['country'].str.lower().str.replace(" ", "_")

# --- Convert string numbers to numeric ---
for col in ['exposed_1km', 'exposed_3km', 'exposed_5km', 'population']:
    country_exposure[col] = pd.to_numeric(country_exposure[col].astype(str).str.replace(',', ''), errors='coerce')

# --- Compute percentage exposed ---
for dist in ['1km', '3km', '5km']:
    country_exposure[f'pct_exposed_{dist}'] = (country_exposure[f'exposed_{dist}'] / country_exposure['population']) * 100

# --- Aggregate social data to country level ---
country_social = social_data.groupby('country', as_index=False)['LDIpc_mean'].mean()
country_social['country'] = country_social['country'].replace({
    'united_states_of_america': 'united_states',
    'usa': 'united_states'
})

# --- Merge datasets ---
merged = country_exposure.merge(country_social, on='country', how='left')
merged = merged.merge(wbg_df, left_on='country', right_on='Economy', how='left')

# --- Combine income groups ---
def combine_income_group(row):
    if row['country'] == 'venezuela':
        return 'Low & Middle income'
    if pd.isnull(row['Income group']):
        return row['Income group']
    if any(x in row['Income group'].lower() for x in ['middle', 'low']):
        return 'Low & Middle income'
    return 'High income'

merged['Income group combined'] = merged.apply(combine_income_group, axis=1)

# --- Define final groups and colors ---
income_groups = ['High income', 'Low & Middle income']
palette = sns.color_palette('tab10', n_colors=len(income_groups))
color_map = dict(zip(income_groups, palette))

plot_merged = merged[merged['Income group combined'].isin(income_groups)].copy()

# --- Create subplots (1 row, 2 columns) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 9), sharey=False)

# --- Set universal font sizes ---
label_fontsize = 18
tick_fontsize = 14
title_fontsize = 20
country_fontsize = 12

for i, group in enumerate(income_groups):
    group_data = plot_merged[plot_merged['Income group combined'] == group]
    ax = axes[i]
    
    # Switched axes: LDIpc on x-axis, % exposed on y-axis
    sns.scatterplot(
        data=group_data,
        x='LDIpc_mean',
        y='pct_exposed_3km',
        color=color_map[group],
        s=150,
        alpha=0.8,
        label=group,
        ax=ax
    )

    # --- Calculate axis limits with small margins ---
    x_min, x_max = group_data['LDIpc_mean'].min(), group_data['LDIpc_mean'].max()
    y_min, y_max = group_data['pct_exposed_3km'].min(), group_data['pct_exposed_3km'].max()
    x_margin = (x_max - x_min) * 0.05 if x_max > x_min else 1
    y_margin = (y_max - y_min) * 0.05 if y_max > y_min else 1
    ax.set_xlim(x_min - x_margin, x_max + x_margin)
    ax.set_ylim(y_min - y_margin, y_max + y_margin)

# --- Country labels (larger font, with manual nudges for specific countries) ---
    for _, row in group_data.iterrows():

        # Default offsets
        x_offset = x_margin * 0.02
        y_offset = y_margin * 0.02

        # Manual nudges for specific countries
        if row['country'] == 'new_zealand':
            x_offset = x_margin * 0.06   # move right
            y_offset = y_margin * -0.6   # keep vertically aligned

        if row['country'] == 'austria':
            x_offset = x_margin * 0.06   # move right
            y_offset = y_margin * -0.6   # keep vertically aligned

        if row['country'] == 'south_sudan':
            x_offset = x_margin * 0.2  # move left
            y_offset = y_margin * 0.6   # small bump up

        if row['country'] == 'ethiopia':
            x_offset = x_margin * 0.5   # move right
            y_offset = y_margin * -0.7   # keep vertically aligned    

        if row['country'] == 'mozambique':
            x_offset = x_margin * 0.9   # move right
            y_offset = y_margin * -0.35   # keep vertically aligned

        if row['country'] == 'sudan':
            x_offset = x_margin * 0.5   # move right
            y_offset = y_margin * -0.7   # keep vertically aligned

        ax.text(
            row['LDIpc_mean'] + x_offset,
            row['pct_exposed_3km'] + y_offset,
            row['country'].replace('_', ' ').title(),
            fontsize=country_fontsize,
            ha='center',
            va='center',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=2)
        )

    # --- Regression line ---
    if len(group_data) > 1:
        sns.regplot(
            data=group_data,
            x='LDIpc_mean',
            y='pct_exposed_3km',
            scatter=False,
            color=color_map[group],
            line_kws={'linewidth': 3},
            ax=ax
        )

    # --- Labels ---
    ax.set_xlabel('Average LDI (country mean)', fontsize=label_fontsize, labelpad=10)
    if i == 0:
        ax.set_ylabel('Population Exposed within 3km (%)', fontsize=label_fontsize, labelpad=10)
    else:
        ax.set_ylabel('')
    ax.set_title(f'{group}', fontsize=title_fontsize, pad=15)

    # --- Tick label size ---
    ax.tick_params(axis='both', which='major', labelsize=tick_fontsize)

# --- Figure  ---
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# --- Save figure ---
# Filename
fig_path = os.path.join(fig_dir, "Figure2_by_income_level_exposure.png")

# Save (high resolution, tight layout)
fig.savefig(
    fig_path,
    dpi=600,            # high quality
    bbox_inches="tight",
    pad_inches=0.2
)

print(f"Figure saved to: {fig_path}")


In [ ]:
## Figure 3

# Standardize country names in shapefile
merged_shapefile_social['ADM0_NAME'] = merged_shapefile_social['ADM0_NAME'].str.lower().str.replace(" ", "_")

# Merge exposure and spatial/social data
merged = adm2_exposed.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# Filter for non-zero, non-null exposure
merged_nonzero = merged[(merged['pct_exposed_3km'] > 0) & merged['pct_exposed_3km'].notna()].copy()

# Convert proportion to percent
merged_nonzero['pct_exposed_3km_percent'] = merged_nonzero['pct_exposed_3km'] * 100

# Bin into 5% intervals (0–5%, 5–10%, ..., up to 100%)
merged_nonzero['pct_exposed_3km_bin'] = pd.cut(
    merged_nonzero['pct_exposed_3km_percent'],
    bins=range(0, 105, 5),
    labels=[f"{i}-{i+5}%" for i in range(0, 100, 5)],
    right=False
)

# Drop rows outside bin range
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_3km_bin'].notna()]

#  WBG income classification
wbg_df['Economy'] = wbg_df['Economy'].str.lower().str.replace(" ", "_")

# Merge classification into main DataFrame
merged_nonzero = merged_nonzero.merge(
    wbg_df,
    left_on='ADM0_NAME',
    right_on='Economy',
    how='left'
)

# --- Clean data as before ---
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_3km'] > 0].copy()
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_3km'].notna()]

countries_to_remove = [
    "austria", "belgium", "chile", "ecuador", "ethiopia", "france", 
    "libya", "mozambique", "saudi_arabia", "south_sudan", "sudan"
]
merged_nonzero = merged_nonzero[~merged_nonzero['country'].str.lower().isin(countries_to_remove)]

# --- Quartile function ---
def assign_quartiles(x):
    try:
        q, bins = pd.qcut(x, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'], retbins=True, duplicates='drop')
        n_bins = len(bins) - 1
        if n_bins < 4:
            labels = [f"Q{i+1}" for i in range(n_bins)]
            q = pd.qcut(x, n_bins, labels=labels, duplicates='drop')
        return q
    except ValueError:
        return pd.Series([pd.NA]*len(x), index=x.index)

merged_nonzero['pct_exposed_3km_quartile_country'] = (
    merged_nonzero.groupby('country')['pct_exposed_3km'].transform(assign_quartiles)
)

# --- Calculate IQR ---
iqr_df = merged_nonzero.groupby('country')['pct_exposed_3km'].quantile([0.25, 0.75]).unstack()
iqr_df.columns = ['Q1', 'Q3']
iqr_df['IQR'] = iqr_df['Q3'] - iqr_df['Q1']

def make_iqr_title(country):
    country_name = country.replace('_', ' ').title()
    if country in iqr_df.index:
        # multiply values by 100 for percentage
        q1 = iqr_df.loc[country, 'Q1'] * 100
        q3 = iqr_df.loc[country, 'Q3'] * 100
        iqr = iqr_df.loc[country, 'IQR'] * 100
        return f"{country_name}\nIQR: {q1:.1f}–{q3:.1f} (Δ={iqr:.1f}%)"
    else:
        return country_name

# --- Plot ---
# --- Define income group color palette ---
income_colors = {
    'High income': '#1f77b4',        # blue
    'Low & Middle income': '#ff7f0e' # orange
}

# --- Ensure 'Income group combined' exists ---
if 'Income group combined' not in merged_nonzero.columns:
    merged_nonzero['Income group combined'] = merged_nonzero['Income group'].apply(
        lambda x: 'Low & Middle income' if pd.isna(x) 
                  else ('High income' if 'high' in str(x).lower() else 'Low & Middle income')
    )

 

# --- Plot boxplots colored by income group ---
g = sns.catplot(
    data=merged_nonzero,
    x='pct_exposed_3km_quartile_country',
    y='LDIpc_mean',
    col='country',
    kind='box',
    col_wrap=5,
    height=4,
    sharey=False,
    palette=income_colors,
    hue='Income group combined',
    dodge=False
)

# --- Add legend with bigger title and labels ---
legend = g._legend
if legend is not None:
    # Set legend title size
    legend.set_title("Income Group", prop={"size": 18, "weight": "bold"})
    # Set legend label sizes
    for text in legend.get_texts():
        text.set_fontsize(18)

# Set custom titles with country bolded, IQR normal weight
for ax, country in zip(g.axes.flat, g.col_names):
    ax.set_title(make_iqr_title(country), fontsize=16, fontweight='bold')

# Increase axis label sizes
g.set_axis_labels('Quartile of % Exposure(3 km)', 'LDI', fontsize=16)

# Adjust spacing between rows to avoid overlap of x-axis labels
g.fig.subplots_adjust(hspace=0.4, top=0.9)

# --- Increase legend size ---
legend = g._legend
if legend is not None:
    legend.set_title("Income Group", prop={"size": 16, "weight": "bold"})
    for text in legend.get_texts():
        text.set_fontsize(14)

# Remove the automatically placed legend
if g._legend is not None:
    g._legend.remove()

# Get handles and labels from one of the subplots (they contain the hue info)
handles, labels = g.axes[0].get_legend_handles_labels()

# Create a new legend centered below all subplots
g.fig.legend(
    handles=handles,
    labels=labels,
    title="Income Group",
    title_fontsize=16,
    fontsize=14,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),  # move below all plots
    ncol=2,                        # side-by-side entries
    frameon=True,                   # show box around legend
    edgecolor='black',              # black border
    facecolor='white'#,              # white background
    #linewidth=1.2
)

plt.show()

# --- Save figure ---
# Filename
fig_path = os.path.join(fig_dir, "Figure3_by_country_exposure.png")

# Save (high resolution, tight layout)
fig.savefig(
    fig_path,
    dpi=600,            # high quality
    bbox_inches="tight",
    pad_inches=0.2
)

print(f"Figure saved to: {fig_path}")


In [ ]:
## Figure 4

# Merge GRDI with exposure
merged = GRDI_data.merge(
    adm2_exposed,
    left_on=['country', 'ID_admin_unit'],
    right_on=['country', 'ID_admin_unit'],
    how='inner'
)

# ── Merge with ADM2 spatial/social data ────────
merged_shapefile_social['ADM0_NAME'] = (
    merged_shapefile_social['ADM0_NAME'].str.lower().str.replace(" ", "_")
)

merged = merged.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# ── Keep only valid data ───────────────────────
merged = merged[(merged['pct_exposed_1km'].notna()) & (merged['GRDI_1km'].notna())].copy()

#  Convert to percent
merged['pct_exposed_1km_percent'] = merged['pct_exposed_1km'] * 100
merged['GRDI_1km_percent'] = merged['GRDI_1km'] * 100

merged['pct_exposed_3km_percent'] = merged['pct_exposed_3km'] * 100
merged['GRDI_3km_percent'] = merged['GRDI_3km'] * 100

merged['pct_exposed_5km_percent'] = merged['pct_exposed_5km'] * 100
merged['GRDI_5km_percent'] = merged['GRDI_5km'] * 100

# Merge classification
merged = merged.merge(
    wbg_df,
    left_on='ADM0_NAME',
    right_on='Economy',
    how='left'
)


# Filter for valid data and GRDI bounds
plot_df = merged[
    (merged['pct_exposed_3km_percent'].notna()) &
    (merged['GRDI_3km'].notna()) &
    (merged['pct_exposed_1km_percent'].notna()) &
    (merged['GRDI_1km'].notna()) &
    (merged['GRDI_3km'].between(0, 100)) &
    (merged['GRDI_1km'].between(0, 100))
].copy()

# Optional: subset to countries with enough data points
#min_obs = 10
#country_counts = plot_df['country'].value_counts()
#valid_countries = country_counts[country_counts >= min_obs].index
#plot_df = plot_df[plot_df['country'].isin(valid_countries)]

# Format country names for titles
plot_df['country_label'] = plot_df['country'].str.replace('_', ' ').str.title()


# Add income group classification
# ---------------------------
high_income = [
    'Australia','Austria','Belgium','Canada','Switzerland','Germany','Denmark','Spain',
    'Finland','France','United Kingdom','Iceland','Ireland','Italy','Japan','Korea, South',
    'Netherlands','Norway','New Zealand','Portugal','Singapore','Sweden','United States',
    'United Arab Emirates','Saudi Arabia'
]

plot_df['income_group'] = plot_df['ADM0_NAME'].apply(
    lambda x: "High income" if x.title().replace('_', ' ') in high_income else "Low and middle income"
)

# Force facet order: High income first
plot_df['income_group'] = pd.Categorical(
    plot_df['income_group'],
    categories=["High income", "Low and middle income"],
    ordered=True
)


#  Reshape data from wide to long format
# ---------------------------
long_df = pd.concat([
    plot_df[['country_label', 'pct_exposed_1km_percent', 'GRDI_1km', 'income_group']]
        .rename(columns={'pct_exposed_1km_percent': 'pct_exposed', 'GRDI_1km': 'GRDI'})
        .assign(buffer='1km'),

    plot_df[['country_label', 'pct_exposed_3km_percent', 'GRDI_3km', 'income_group']]
        .rename(columns={'pct_exposed_3km_percent': 'pct_exposed', 'GRDI_3km': 'GRDI'})
        .assign(buffer='3km'),

    plot_df[['country_label', 'pct_exposed_5km_percent', 'GRDI_5km', 'income_group']]
        .rename(columns={'pct_exposed_5km_percent': 'pct_exposed', 'GRDI_5km': 'GRDI'})
        .assign(buffer='5km')
])



# Faceted scatter plot by income group
# ---------------------------
g = sns.lmplot(
    data=long_df,
    x='pct_exposed',
    y='GRDI',
    hue='buffer',
    col='income_group',
    col_order=["High income", "Low and middle income"],
    height=6,
    aspect=1.2,
    scatter_kws={'alpha': 0.35, 's': 2},  # << even smaller points
    line_kws={'linewidth': 2},
    ci=95,
    facet_kws={'sharex': False, 'sharey': True}
)

# Axis labels and titles
g.set_axis_labels('% Exposed', 'GRDI', fontsize=18)
g.set_titles('{col_name}', size=18)
plt.subplots_adjust(top=0.9)

# Set axis limits
for ax in g.axes.flat:
    ax.set_xlim(0, 100)  # x-axis from 0 to 100
    ax.set_ylim(20, 80)  # y-axis from 20 to 80
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)

# Legend adjustments
# ---------------------------
g._legend.remove()

buffer_colors = {
    b: c for b, c in zip(
        long_df['buffer'].unique(),
        sns.color_palette(n_colors=len(long_df['buffer'].unique()))
    )
}

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=b,
           markerfacecolor=buffer_colors[b], markersize=12)
    for b in buffer_colors
]

g.fig.legend(
    handles=legend_elements,
    loc='lower center',
    ncol=len(buffer_colors),
    fontsize=16,
    frameon=True,
    framealpha=0.9,
    bbox_to_anchor=(0.5, -0.12)
)

plt.show()

# --- Save figure ---
# Filename
fig_path = os.path.join(fig_dir, "Figure4_by_region_GRDI.png")

# Save (high resolution, tight layout)
fig.savefig(
    fig_path,
    dpi=600,            # high quality
    bbox_inches="tight",
    pad_inches=0.2
)

print(f"Figure saved to: {fig_path}")



In [ ]:
## Supplementary Figures

## Figure S1
# ----------------------------------------------------
# Standardize country names in shapefile
# ----------------------------------------------------
merged_shapefile_social['ADM0_NAME'] = (
    merged_shapefile_social['ADM0_NAME']
    .str.lower()
    .str.replace(" ", "_")
)

# ----------------------------------------------------
# Merge exposure and spatial/social data at ADM2 level
# ----------------------------------------------------
merged = adm2_exposed.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# ----------------------------------------------------
# Ensure we have a GeoDataFrame and geometry column
# ----------------------------------------------------
gdf_plot = gpd.GeoDataFrame(merged, geometry='geometry', crs=4326)

# Reproject to Web Mercator for contextily basemap
gdf_plot = gdf_plot.to_crs(epsg=3857)

# ----------------------------------------------------
# Income group classification
# ----------------------------------------------------
high_income = [
    'Australia','Austria','Belgium','Canada','Switzerland','Germany','Denmark','Spain',
    'Finland','France','United Kingdom','Iceland','Ireland','Italy','Japan','Korea, South',
    'Netherlands','Norway','New Zealand','Portugal','Singapore','Sweden','United States',
    'United Arab Emirates','Saudi Arabia'
]

# Standardize ADM0_NAME for comparison
gdf_plot['ADM0_NAME_clean'] = (
    gdf_plot['ADM0_NAME']
    .astype(str)
    .str.strip()
    .str.replace('_', ' ')
    .str.title()
)

# Assign income group; fill any NaNs
gdf_plot['income_group'] = gdf_plot['ADM0_NAME_clean'].apply(
    lambda x: "High income" if x in high_income else "Low and middle income"
)

# Ensure categorical order
gdf_plot['income_group'] = pd.Categorical(
    gdf_plot['income_group'],
    categories=["High income", "Low and middle income"],
    ordered=True
)

# ----------------------------------------------------
# Define colors
# ----------------------------------------------------
income_colors = {
    'High income': '#1f77b4',        # blue
    'Low and middle income': '#ff7f0e' # orange
}
gdf_plot = gdf_plot[gdf_plot['exposed_1km'].notna()]

# Fill any potential NaNs in color mapping
gdf_plot['plot_color'] = gdf_plot['income_group'].map(income_colors).astype(str).fillna('#d3d3d3')

# ----------------------------------------------------
# Plot
# ----------------------------------------------------
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

gdf_plot.plot(
    ax=ax,
    color=gdf_plot['plot_color'],
    edgecolor='black',
    linewidth=0.2
)

# Add grey basemap
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas)

# Legend
handles = [
    Line2D([0], [0], marker='s', color='w',
           markerfacecolor=color, label=group,
           markersize=10, linestyle='')
    for group, color in income_colors.items()
]

ax.legend(
    handles=handles,
    title='Income Group',
    loc='lower left',
    fontsize=12,
    title_fontsize=14
)

ax.set_axis_off()
plt.tight_layout()
plt.show()

# ----------------------- Counting ADM2 units
# Count unique ID_admin_unit per income group
unique_counts = gdf_plot.groupby('income_group')['ID_admin_unit'].nunique().reset_index()

# Rename column for clarity
unique_counts.rename(columns={'ID_admin_unit': 'num_unique_admin_units'}, inplace=True)

print(unique_counts)

In [ ]:
## Supplementary Figures
##  Figure S2

# Load the two CSVs
land_df = pd.read_csv("/Users/larasch/Documents/UCB_postdoc/Research/global_oil_gas/results/land_area_exposed_to_hazards.csv")
wells_df = pd.read_csv("/Users/larasch/Documents/UCB_postdoc/Research/global_oil_gas/results/well_counts_by_adm2.csv")

# Merge on ID_admin_unit = ADM2_CODE and drop rows with NA after merge
merged = pd.merge(
    land_df,
    wells_df,
    left_on="ID_admin_unit",
    right_on="ADM2_CODE",
    how="left"
)
merged = merged.dropna(subset=["ID_admin_unit", "ADM2_CODE", "well_count", "population"])

# Create new variable: well_count / population
merged["wells_per_area"] = merged["well_count"] / merged["population"]

# Load WBG classification data
wbg_df['Economy'] = wbg_df['Economy'].str.lower().str.replace(" ", "_")

# Merge WBG classification into merged DataFrame
merged = merged.merge(
    wbg_df,
    left_on='country_x',
    right_on='Economy',
    how='left'
)

# --- Combine lower, middle, and Venezuela into 'Middle/Low income' ---
def combine_income_group(row):
    if row['country_x'].lower() == 'venezuela':
        return 'Middle/Low income'
    if pd.isnull(row['Income group']):
        return row['Income group']
    if 'low' in row['Income group'].lower() or 'middle' in row['Income group'].lower():
        return 'Low & Middle income'
    return 'High income'

merged['Income group combined'] = merged.apply(combine_income_group, axis=1)

# Prepare data for binscatter
plot_df = merged.dropna(subset=['pct_exposed_3km', 'wells_per_area', 'Income group combined']).copy()
plot_df = plot_df[plot_df['wells_per_area'] <= 2.5]

# Only plot High and Middle/Low income in desired order
income_groups = ['High income', 'Low & Middle income']
plot_df['Income group combined'] = pd.Categorical(plot_df['Income group combined'], categories=income_groups, ordered=True)
plot_df = plot_df[plot_df['Income group combined'].isin(income_groups)]

# Create bins for x-axis
num_bins = 20
plot_df['x_bin'] = pd.cut(plot_df['pct_exposed_3km'], bins=num_bins)

# Aggregate mean and standard error per bin and income group
binned_df = plot_df.groupby(['x_bin', 'Income group combined']).agg(
    pct_exposed_mean=('pct_exposed_3km', 'mean'),
    wells_per_area_mean=('wells_per_area', 'mean'),
    wells_per_area_se=('wells_per_area', lambda x: x.std() / np.sqrt(len(x)))  # standard error
).reset_index()

# Plot binscatter with shaded confidence intervals
plt.figure(figsize=(9, 7))
palette = sns.color_palette("tab10", n_colors=len(income_groups))

for i, group in enumerate(income_groups):
    group_df = binned_df[binned_df['Income group combined'] == group]
    plt.plot(group_df['pct_exposed_mean'], group_df['wells_per_area_mean'], marker='o', color=palette[i], label=group)
    plt.fill_between(
        group_df['pct_exposed_mean'],
        group_df['wells_per_area_mean'] - group_df['wells_per_area_se'],
        group_df['wells_per_area_mean'] + group_df['wells_per_area_se'],
        color=palette[i],
        alpha=0.2
    )

plt.xlabel("Proportion ADM2 Exposed (3km)")
plt.ylabel("Average Wells per Area")
#plt.title("Binscatter: % Exposed vs Wells per Grid \nby WBG Income Group (Middle + Low combined)")
plt.legend(title="Income Group", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
## Figure S3
social_data['country_location_name'] = social_data['country_location_name'].replace(
    {'Venezuela (Bolivarian Republic of)': 'Venezuela'}
)
social_data['country'] = social_data['country_location_name'].str.lower().str.replace(" ", "_")
country_exposure['country'] = country_exposure['country'].str.lower().str.replace(" ", "_")
country_exposure['exposed_1km'] = pd.to_numeric(country_exposure['exposed_1km'].astype(str).str.replace(',', ''), errors='coerce')
country_exposure['exposed_3km'] = pd.to_numeric(country_exposure['exposed_3km'].astype(str).str.replace(',', ''), errors='coerce')
country_exposure['exposed_5km'] = pd.to_numeric(country_exposure['exposed_5km'].astype(str).str.replace(',', ''), errors='coerce')
country_exposure['population'] = pd.to_numeric(country_exposure['population'].astype(str).str.replace(',', ''), errors='coerce')
country_exposure['pct_exposed_1km'] = (country_exposure['exposed_1km'] / country_exposure['population']) * 100
country_exposure['pct_exposed_3km'] = (country_exposure['exposed_3km'] / country_exposure['population']) * 100
country_exposure['pct_exposed_5km'] = (country_exposure['exposed_5km'] / country_exposure['population']) * 100

# Aggregate social data to country level (mean LDIpc_mean)
country_social = social_data.groupby('country', as_index=False)['LDIpc_mean'].mean()
country_social['country'] = country_social['country'].replace({
    'united_states_of_america': 'united_states',
    'usa': 'united_states'
})

# --- Merge exposure and social data ---
merged = country_exposure.merge(country_social, left_on='country', right_on='country', how='left')
merged = merged.merge(wbg_df, left_on='country', right_on='Economy', how='left')

# --- Combine income groups into "High income" vs "Low and Middle income" ---
def combine_income_group(row):
    if row['country'] == 'venezuela':
        return 'Low and Middle income'
    if pd.isnull(row['Income group']):
        return row['Income group']
    if 'low' in row['Income group'].lower() or 'middle' in row['Income group'].lower():
        return 'Low and Middle income'
    return 'High income'

merged['Income group combined'] = merged.apply(combine_income_group, axis=1)

# Use the combined income group for filtering and plotting
income_groups = ['High income', 'Low and Middle income']
plot_merged = merged[merged['Income group combined'].isin(income_groups)].copy()

exposures = {
    'pct_exposed_1km': ('1 km', 'blue'),
    'pct_exposed_3km': ('3 km', 'orange'),
    'pct_exposed_5km': ('5 km', 'green')
}

fig, axes = plt.subplots(3, 2, figsize=(14, 14), sharey=False, constrained_layout=True)  # 2 cols now

for row_idx, (exposure_var, (label, color)) in enumerate(exposures.items()):
    for col_idx, group in enumerate(income_groups):
        group_data = plot_merged[plot_merged['Income group combined'] == group]
        ax = axes[row_idx, col_idx]

        # Scatter plot
        sns.scatterplot(
            data=group_data,
            x=exposure_var,
            y='LDIpc_mean',
            color=color,
            s=120,
            alpha=0.8,
            ax=ax
        )

        # Regression line
        if len(group_data) > 1:
            sns.regplot(
                data=group_data,
                x=exposure_var,
                y='LDIpc_mean',
                scatter=False,
                color=color,
                line_kws={'linewidth': 2},
                ax=ax
            )

        # Axis limits for overlap detection
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()

        # Country labels
        coords = []
        for _, row in group_data.iterrows():
            x, y = row[exposure_var], row['LDIpc_mean']
            country_name = row['country'].replace('_', ' ').title()

            near_axis = (abs(x - xlim[0]) < 0.02*(xlim[1]-xlim[0])) or \
                        (abs(x - xlim[1]) < 0.02*(xlim[1]-xlim[0])) or \
                        (abs(y - ylim[0]) < 0.02*(ylim[1]-ylim[0])) or \
                        (abs(y - ylim[1]) < 0.02*(ylim[1]-ylim[0]))
            close_to_other = any(np.hypot(x - xx, y - yy) < 0.5 for xx, yy in coords)
            coords.append((x, y))

            if near_axis or close_to_other:
                ax.annotate(
                    country_name,
                    xy=(x, y),
                    xytext=(10, 10),
                    textcoords='offset points',
                    fontsize=7,
                    ha='left',
                    va='bottom',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8)
                )
            else:
                ax.text(
                    x, y, country_name,
                    fontsize=7,
                    ha='right', va='bottom',
                    bbox=dict(facecolor='white', edgecolor='none', alpha=0.9, pad=1)
                )

        # Axis labels
        ax.set_xlabel('Population Exposed (%)')
        if col_idx == 0:
            ax.set_ylabel('Average LDIpc (country mean)')
        else:
            ax.set_ylabel('')

        # Only add column titles in the top row
        if row_idx == 0:
            ax.set_title(group)
        else:
            ax.set_title("")

        # Add row labels (1 km / 3 km / 5 km) on the left
        if col_idx == 0:
            ax.annotate(
                label, xy=(0, 0.5), xytext=(-ax.yaxis.labelpad - 40, 0),
                xycoords=ax.yaxis.label, textcoords='offset points',
                size=12, ha='right', va='center', rotation=90, color=color
            )

        # Force x-axis to start at 0
        ax.set_xlim(left=0)

        # Small margins for labels
        ax.margins(x=0.05, y=0.05)

#plt.suptitle('Population Exposed vs. Average Lag-distributed Income by Income Group and Buffer Distance', fontsize=16)
plt.show()


In [ ]:
# Figure S4

# Standardize country names in shapefile
merged_shapefile_social['ADM0_NAME'] = merged_shapefile_social['ADM0_NAME'].str.lower().str.replace(" ", "_")

# Merge exposure and spatial/social data
merged = adm2_exposed.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# Filter for non-zero, non-null exposure
merged_nonzero = merged[(merged['pct_exposed_1km'] > 0) & merged['pct_exposed_1km'].notna()].copy()

# Convert proportion to percent
merged_nonzero['pct_exposed_1km_percent'] = merged_nonzero['pct_exposed_1km'] * 100

# Bin into 5% intervals (0–5%, 5–10%, ..., up to 100%)
merged_nonzero['pct_exposed_1km_bin'] = pd.cut(
    merged_nonzero['pct_exposed_1km_percent'],
    bins=range(0, 105, 5),
    labels=[f"{i}-{i+5}%" for i in range(0, 100, 5)],
    right=False
)

# Drop rows outside bin range
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_1km_bin'].notna()]

#  WBG income classification
wbg_df['Economy'] = wbg_df['Economy'].str.lower().str.replace(" ", "_")

# Merge classification into main DataFrame
merged_nonzero = merged_nonzero.merge(
    wbg_df,
    left_on='ADM0_NAME',
    right_on='Economy',
    how='left'
)

# --- Clean data as before ---
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_1km'] > 0].copy()
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_1km'].notna()]

countries_to_remove = [
    "austria", "belgium", "chile", "ecuador", "ethiopia", "france", 
    "libya", "mozambique", "saudi_arabia", "south_sudan", "sudan"
]
merged_nonzero = merged_nonzero[~merged_nonzero['country'].str.lower().isin(countries_to_remove)]

# --- Quartile function ---
def assign_quartiles(x):
    try:
        q, bins = pd.qcut(x, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'], retbins=True, duplicates='drop')
        n_bins = len(bins) - 1
        if n_bins < 4:
            labels = [f"Q{i+1}" for i in range(n_bins)]
            q = pd.qcut(x, n_bins, labels=labels, duplicates='drop')
        return q
    except ValueError:
        return pd.Series([pd.NA]*len(x), index=x.index)

merged_nonzero['pct_exposed_1km_quartile_country'] = (
    merged_nonzero.groupby('country')['pct_exposed_1km'].transform(assign_quartiles)
)

# --- Calculate IQR ---
iqr_df = merged_nonzero.groupby('country')['pct_exposed_1km'].quantile([0.25, 0.75]).unstack()
iqr_df.columns = ['Q1', 'Q3']
iqr_df['IQR'] = iqr_df['Q3'] - iqr_df['Q1']

def make_iqr_title(country):
    country_name = country.replace('_', ' ').title()
    if country in iqr_df.index:
        # multiply values by 100 for percentage
        q1 = iqr_df.loc[country, 'Q1'] * 100
        q3 = iqr_df.loc[country, 'Q3'] * 100
        iqr = iqr_df.loc[country, 'IQR'] * 100
        return f"{country_name}\nIQR: {q1:.1f}–{q3:.1f} (Δ={iqr:.1f}%)"
    else:
        return country_name

# --- Plot ---
# --- Define income group color palette ---
income_colors = {
    'High income': '#1f77b4',        # blue
    'Low & Middle income': '#ff7f0e' # orange
}

# --- Ensure 'Income group combined' exists ---
if 'Income group combined' not in merged_nonzero.columns:
    merged_nonzero['Income group combined'] = merged_nonzero['Income group'].apply(
        lambda x: 'Low & Middle income' if pd.isna(x) 
                  else ('High income' if 'high' in str(x).lower() else 'Low & Middle income')
    )

 

# --- Plot boxplots colored by income group ---
g = sns.catplot(
    data=merged_nonzero,
    x='pct_exposed_1km_quartile_country',
    y='LDIpc_mean',
    col='country',
    kind='box',
    col_wrap=5,
    height=4,
    sharey=False,
    palette=income_colors,
    hue='Income group combined',
    dodge=False
)

# --- Add legend with bigger title and labels ---
legend = g._legend
if legend is not None:
    # Set legend title size
    legend.set_title("Income Group", prop={"size": 18, "weight": "bold"})
    # Set legend label sizes
    for text in legend.get_texts():
        text.set_fontsize(18)

# Set custom titles with country bolded, IQR normal weight
for ax, country in zip(g.axes.flat, g.col_names):
    ax.set_title(make_iqr_title(country), fontsize=16, fontweight='bold')

# Increase axis label sizes
g.set_axis_labels('Quartile of % Exposure (1 km)', 'LDI', fontsize=16)

# Adjust spacing between rows to avoid overlap of x-axis labels
g.fig.subplots_adjust(hspace=0.4, top=0.9)

# --- Increase legend size ---
legend = g._legend
if legend is not None:
    legend.set_title("Income Group", prop={"size": 16, "weight": "bold"})
    for text in legend.get_texts():
        text.set_fontsize(14)

# Remove the automatically placed legend
if g._legend is not None:
    g._legend.remove()

# Get handles and labels from one of the subplots (they contain the hue info)
handles, labels = g.axes[0].get_legend_handles_labels()

# Create a new legend centered below all subplots
g.fig.legend(
    handles=handles,
    labels=labels,
    title="Income Group",
    title_fontsize=16,
    fontsize=14,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),  # move below all plots
    ncol=2,                        # side-by-side entries
    frameon=True,                   # show box around legend
    edgecolor='black',              # black border
    facecolor='white'#,              # white background
    #linewidth=1.2
)

plt.show()




In [ ]:
# Figure S5

# Standardize country names in shapefile
merged_shapefile_social['ADM0_NAME'] = merged_shapefile_social['ADM0_NAME'].str.lower().str.replace(" ", "_")

# Merge exposure and spatial/social data
merged = adm2_exposed.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# Filter for non-zero, non-null exposure
merged_nonzero = merged[(merged['pct_exposed_5km'] > 0) & merged['pct_exposed_5km'].notna()].copy()

# Convert proportion to percent
merged_nonzero['pct_exposed_5km_percent'] = merged_nonzero['pct_exposed_5km'] * 100

# Bin into 5% intervals (0–5%, 5–10%, ..., up to 100%)
merged_nonzero['pct_exposed_5km_bin'] = pd.cut(
    merged_nonzero['pct_exposed_5km_percent'],
    bins=range(0, 105, 5),
    labels=[f"{i}-{i+5}%" for i in range(0, 100, 5)],
    right=False
)

# Drop rows outside bin range
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_5km_bin'].notna()]

#  WBG income classification
wbg_df['Economy'] = wbg_df['Economy'].str.lower().str.replace(" ", "_")

# Merge classification into main DataFrame
merged_nonzero = merged_nonzero.merge(
    wbg_df,
    left_on='ADM0_NAME',
    right_on='Economy',
    how='left'
)

# --- Clean data as before ---
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_5km'] > 0].copy()
merged_nonzero = merged_nonzero[merged_nonzero['pct_exposed_5km'].notna()]

countries_to_remove = [
    "austria", "belgium", "chile", "ecuador", "ethiopia", "france", 
    "libya", "mozambique", "saudi_arabia", "south_sudan", "sudan"
]
merged_nonzero = merged_nonzero[~merged_nonzero['country'].str.lower().isin(countries_to_remove)]

# --- Quartile function ---
def assign_quartiles(x):
    try:
        q, bins = pd.qcut(x, 4, labels=['Q1', 'Q2', 'Q3', 'Q4'], retbins=True, duplicates='drop')
        n_bins = len(bins) - 1
        if n_bins < 4:
            labels = [f"Q{i+1}" for i in range(n_bins)]
            q = pd.qcut(x, n_bins, labels=labels, duplicates='drop')
        return q
    except ValueError:
        return pd.Series([pd.NA]*len(x), index=x.index)

merged_nonzero['pct_exposed_5km_quartile_country'] = (
    merged_nonzero.groupby('country')['pct_exposed_5km'].transform(assign_quartiles)
)

# --- Calculate IQR ---
iqr_df = merged_nonzero.groupby('country')['pct_exposed_5km'].quantile([0.25, 0.75]).unstack()
iqr_df.columns = ['Q1', 'Q3']
iqr_df['IQR'] = iqr_df['Q3'] - iqr_df['Q1']

def make_iqr_title(country):
    country_name = country.replace('_', ' ').title()
    if country in iqr_df.index:
        # multiply values by 100 for percentage
        q1 = iqr_df.loc[country, 'Q1'] * 100
        q3 = iqr_df.loc[country, 'Q3'] * 100
        iqr = iqr_df.loc[country, 'IQR'] * 100
        return f"{country_name}\nIQR: {q1:.1f}–{q3:.1f} (Δ={iqr:.1f}%)"
    else:
        return country_name

# --- Plot ---
# --- Define income group color palette ---
income_colors = {
    'High income': '#1f77b4',        # blue
    'Low & Middle income': '#ff7f0e' # orange
}

# --- Ensure 'Income group combined' exists ---
if 'Income group combined' not in merged_nonzero.columns:
    merged_nonzero['Income group combined'] = merged_nonzero['Income group'].apply(
        lambda x: 'Low & Middle income' if pd.isna(x) 
                  else ('High income' if 'high' in str(x).lower() else 'Low & Middle income')
    )

 

# --- Plot boxplots colored by income group ---
g = sns.catplot(
    data=merged_nonzero,
    x='pct_exposed_5km_quartile_country',
    y='LDIpc_mean',
    col='country',
    kind='box',
    col_wrap=5,
    height=4,
    sharey=False,
    palette=income_colors,
    hue='Income group combined',
    dodge=False
)

# --- Add legend with bigger title and labels ---
legend = g._legend
if legend is not None:
    # Set legend title size
    legend.set_title("Income Group", prop={"size": 18, "weight": "bold"})
    # Set legend label sizes
    for text in legend.get_texts():
        text.set_fontsize(18)

# Set custom titles with country bolded, IQR normal weight
for ax, country in zip(g.axes.flat, g.col_names):
    ax.set_title(make_iqr_title(country), fontsize=16, fontweight='bold')

# Increase axis label sizes
g.set_axis_labels('Quartile of % Exposure (5 km)', 'LDI', fontsize=16)

# Adjust spacing between rows to avoid overlap of x-axis labels
g.fig.subplots_adjust(hspace=0.4, top=0.9)

# --- Increase legend size ---
legend = g._legend
if legend is not None:
    legend.set_title("Income Group", prop={"size": 16, "weight": "bold"})
    for text in legend.get_texts():
        text.set_fontsize(14)

# Remove the automatically placed legend
if g._legend is not None:
    g._legend.remove()

# Get handles and labels from one of the subplots (they contain the hue info)
handles, labels = g.axes[0].get_legend_handles_labels()

# Create a new legend centered below all subplots
g.fig.legend(
    handles=handles,
    labels=labels,
    title="Income Group",
    title_fontsize=16,
    fontsize=14,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),  # move below all plots
    ncol=2,                        # side-by-side entries
    frameon=True,                   # show box around legend
    edgecolor='black',              # black border
    facecolor='white'#,              # white background
    #linewidth=1.2
)

plt.show()




In [ ]:
# Figure S6
summary_path = results_dir / "ogd_adm2_pop_exposed.csv"
summary_df = pd.read_csv(summary_path)

merged_shapefile_social['ADM0_NAME'] = merged_shapefile_social['ADM0_NAME'].str.lower().str.replace(" ", "_")

merged = summary_df.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# Remove rows where pct_exposed_3km is zero or NaN
merged_nonzero = merged[(merged['pct_exposed_3km'] > 0) & (merged['pct_exposed_3km'].notna())].copy()

# Merge WBG classification
wbg_df['Economy'] = wbg_df['Economy'].str.lower().str.replace(" ", "_")
merged_nonzero = merged_nonzero.merge(
    wbg_df,
    left_on='ADM0_NAME',
    right_on='Economy',
    how='left'
)

# Combine income groups into High income vs Low and Middle income
def combine_income_group(row):
    if row['ADM0_NAME'] == 'venezuela':
        return 'Low and Middle income'
    if pd.isnull(row['Income group']):
        return row['Income group']
    if 'low' in row['Income group'].lower() or 'middle' in row['Income group'].lower():
        return 'Low and Middle income'
    return 'High income'

merged_nonzero['Income group combined'] = merged_nonzero.apply(combine_income_group, axis=1)

# Set order
income_order = ['High income', 'Low and Middle income']
merged_nonzero['Income group combined'] = pd.Categorical(
    merged_nonzero['Income group combined'],
    categories=income_order,
    ordered=True
)

# Define quartiles within each income group
def assign_group_quartiles(df):
    df = df.copy()
    try:
        df['pct_exposed_3km_quartile'] = pd.qcut(df['pct_exposed_3km'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    except ValueError:
        df['pct_exposed_3km_quartile'] = pd.NA
    return df

merged_nonzero = merged_nonzero.groupby('Income group combined', group_keys=False).apply(assign_group_quartiles)

# Faceted boxplot with custom colors
import seaborn as sns
import matplotlib.pyplot as plt

# --- keep all your data prep code from before this unchanged ---

# Define custom colors
color_map = {
    'High income': '#1f77b4',          # blue
    'Low and Middle income': '#ff7f0e' # orange
}

# Create figure with one subplot per income group
income_order = ['High income', 'Low and Middle income']
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, income_group in zip(axes, income_order):
    subset = merged_nonzero[merged_nonzero['Income group combined'] == income_group]
    sns.boxplot(
        data=subset,
        x='pct_exposed_3km_quartile',
        y='LDIpc_mean',
        color=color_map[income_group],
        ax=ax
    )
    ax.set_title(income_group)
    ax.set_xlabel('Quartile of % Exposed (3km) within income group')
    ax.set_ylabel('LDIpc Mean')

#fig.suptitle('LDIpc Mean by Within-Income Group Quartile of % Exposed (3km) and WBG Classification', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
## Figure S7

# Merge GRDI with exposure
merged = GRDI_data.merge(
    adm2_exposed,
    left_on=['country', 'ID_admin_unit'],
    right_on=['country', 'ID_admin_unit'],
    how='inner'
)

# ── Merge with ADM2 spatial/social data ────────
merged_shapefile_social['ADM0_NAME'] = (
    merged_shapefile_social['ADM0_NAME'].str.lower().str.replace(" ", "_")
)

merged = merged.merge(
    merged_shapefile_social,
    left_on=['country', 'ID_admin_unit'],
    right_on=['ADM0_NAME', 'ADM2_CODE'],
    how='left'
)

# ── Keep only valid data ───────────────────────
merged = merged[(merged['pct_exposed_1km'].notna()) & (merged['GRDI_1km'].notna())].copy()

#  Convert to percent
merged['pct_exposed_1km_percent'] = merged['pct_exposed_1km'] * 100
merged['GRDI_1km_percent'] = merged['GRDI_1km'] * 100

merged['pct_exposed_3km_percent'] = merged['pct_exposed_3km'] * 100
merged['GRDI_3km_percent'] = merged['GRDI_3km'] * 100

merged['pct_exposed_5km_percent'] = merged['pct_exposed_5km'] * 100
merged['GRDI_5km_percent'] = merged['GRDI_5km'] * 100

# Merge classification
merged = merged.merge(
    wbg_df,
    left_on='ADM0_NAME',
    right_on='Economy',
    how='left'
)


# Filter for valid data and GRDI bounds
plot_df = merged[
    (merged['pct_exposed_3km_percent'].notna()) &
    (merged['GRDI_3km'].notna()) &
    (merged['pct_exposed_1km_percent'].notna()) &
    (merged['GRDI_1km'].notna()) &
    (merged['GRDI_3km'].between(0, 100)) &
    (merged['GRDI_1km'].between(0, 100))
].copy()

# Optional: subset to countries with enough data points
min_obs = 10
country_counts = plot_df['country'].value_counts()
valid_countries = country_counts[country_counts >= min_obs].index
plot_df = plot_df[plot_df['country'].isin(valid_countries)]

# Format country names for titles
plot_df['country_label'] = plot_df['country'].str.replace('_', ' ').str.title()

# Reshape data from wide to long format for 1km, 3km, 5km
long_df = pd.concat([
    plot_df[['country_label', 'pct_exposed_1km_percent', 'GRDI_1km']].rename(
        columns={'pct_exposed_1km_percent': 'pct_exposed', 'GRDI_1km': 'GRDI'}
    ).assign(buffer='1km'),
    plot_df[['country_label', 'pct_exposed_3km_percent', 'GRDI_3km']].rename(
        columns={'pct_exposed_3km_percent': 'pct_exposed', 'GRDI_3km': 'GRDI'}
    ).assign(buffer='3km'),
    plot_df[['country_label', 'pct_exposed_5km_percent', 'GRDI_5km']].rename(
        columns={'pct_exposed_5km_percent': 'pct_exposed', 'GRDI_5km': 'GRDI'}
    ).assign(buffer='5km')
])

# Faceted scatter plot with regression lines for all three buffers
# --- Faceted scatter plot with regression lines ---
g = sns.lmplot(
    data=long_df,
    x='pct_exposed',
    y='GRDI',
    hue='buffer',
    col='country_label',
    col_wrap=4,
    height=4,
    scatter_kws={'alpha': 0.4, 's': 10},  
    line_kws={'linewidth': 2},
    facet_kws={'sharex': False, 'sharey': True}
)

# Axis labels and titles
g.set_axis_labels('% Exposed', 'GRDI', fontsize=18)
g.set_titles('{col_name}', size=18)
plt.subplots_adjust(top=0.92)
#g.fig.suptitle('Relationship Between Population OG Well Exposure and Relative Deprivation (GRDI) by Distance from Well', fontsize=22)
plt.ylim(0, 100)

# Tick labels
for ax in g.axes.flat:
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)

# --- Adjust legend ---
# Remove existing legend
g._legend.remove()

# Create a new legend under the plot
from matplotlib.lines import Line2D

# Map buffer to colors (Seaborn default)
buffer_colors = {b: c for b, c in zip(long_df['buffer'].unique(), sns.color_palette(n_colors=len(long_df['buffer'].unique())))}

legend_elements = [Line2D([0], [0], marker='o', color='w', label=b,
                          markerfacecolor=buffer_colors[b], markersize=12) for b in buffer_colors]

g.fig.legend(handles=legend_elements, loc='lower center', ncol=len(buffer_colors), fontsize=16, frameon=True, framealpha=0.9, bbox_to_anchor=(0.5, -0.03))

plt.show()
